# Chilbolton Climatology — Student 4: Wind Speed & Direction

This is **Student 4's notebook** as part of a group project on the climatology of Chilbolton Observatory.
The four students are each analysing a different meteorological variable:
- **Student 1**: Rainfall
- **Student 2**: Air temperature and relative humidity
- **Student 3**: Surface pressure
- **Student 4 (you)**: Wind speed and direction

## Learning Objectives
By the end of this notebook you should be able to:
- Load and quality-control wind data from NetCDF files
- Analyse the statistical distribution of wind speeds
- Produce a wind rose showing the combined distribution of speed and direction
- Identify seasonal differences in wind behaviour at Chilbolton

## Instrument Information
Wind data at Chilbolton are measured by the NCAS sonic anemometer.
- `wind_speed` is stored in **m s⁻¹** (metres per second)
- `wind_from_direction` is stored in **degrees** (0° = wind from North, 90° = from East, etc.)
- QC flags: `qc_flag_wind_speed` and `qc_flag_wind_direction` (value 1 = good)

## Setup — Run This First

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.cm as cm
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import netCDF4 as nc4
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 200)
print('Libraries loaded.')

In [ ]:
def load_wind(root: str, qc_good_only: bool = True) -> pd.DataFrame:
    """
    Load all wind NetCDF files from root and return a time-indexed DataFrame.

    Columns returned:
        time      : UTC timestamp (pandas Timestamp)
        speed     : wind speed in m/s
        direction : wind-from direction in degrees (0=N, 90=E, 180=S, 270=W)
    """
    files = sorted(Path(root).rglob('*.nc'))
    if not files:
        raise FileNotFoundError(f'No .nc files found under {root}')

    chunks = []
    for f in files:
        with nc4.Dataset(str(f)) as nc:
            unix      = nc.variables['time'][:].data.copy().astype(np.float64)
            speed     = nc.variables['wind_speed'][:].data.copy().astype(np.float64)
            direction = nc.variables['wind_from_direction'][:].data.copy().astype(np.float64)
            qc_spd    = nc.variables['qc_flag_wind_speed'][:].data.copy().astype(np.int8)
            qc_dir    = nc.variables['qc_flag_wind_direction'][:].data.copy().astype(np.int8)

        if qc_good_only:
            speed[(qc_spd != 0) & (qc_spd != 1)] = np.nan
            direction[(qc_dir != 0) & (qc_dir != 1)] = np.nan

        chunks.append(pd.DataFrame({'unix': unix, 'speed': speed, 'direction': direction}))

    df = pd.concat(chunks, ignore_index=True).sort_values('unix').reset_index(drop=True)
    df['time'] = pd.to_datetime(df['unix'], unit='s', utc=True)
    return df.drop(columns='unix')


def wind_rose(speed: np.ndarray, direction: np.ndarray,
              n_dir: int = 16, speed_bins: list | None = None,
              ax: plt.Axes | None = None, title: str = 'Wind Rose') -> plt.Axes:
    """
    Plot a wind rose on a polar axis.

    Parameters
    ----------
    speed      : wind speed array (m/s)
    direction  : wind-from direction array (degrees, 0=N)
    n_dir      : number of directional sectors (default 16)
    speed_bins : bin edges for speed categories (default 0,2,5,10,15,np.inf)
    ax         : existing polar axes (created if None)
    title      : plot title
    """
    if speed_bins is None:
        speed_bins = [0, 2, 5, 10, 15, np.inf]
    speed_labels = ['0–2', '2–5', '5–10', '10–15', '>15']
    colours = cm.Blues(np.linspace(0.3, 0.95, len(speed_bins) - 1))

    # Bin directions into equal sectors
    dir_bin_edges = np.linspace(0, 360, n_dir + 1)
    dir_width = 2 * np.pi / n_dir
    # Shift so 0° is North at top (polar theta=0 = East, so offset by 90°)
    theta = np.deg2rad(dir_bin_edges[:-1] + dir_bin_edges[1:]) / 2  # sector centres

    # Drop NaNs
    mask = ~(np.isnan(speed) | np.isnan(direction))
    spd = speed[mask]
    drn = direction[mask] % 360

    # Assign each observation to a direction sector
    dir_bins_idx = np.digitize(drn, dir_bin_edges[1:-1])  # 0 .. n_dir-1

    n_total = len(spd)

    if ax is None:
        fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(7, 7))

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)  # clockwise

    bottom = np.zeros(n_dir)
    for k, (lo, hi) in enumerate(zip(speed_bins[:-1], speed_bins[1:])):
        in_bin = (spd >= lo) & (spd < hi)
        freqs = np.array([
            np.sum((dir_bins_idx == j) & in_bin) / n_total * 100
            for j in range(n_dir)
        ])
        ax.bar(np.deg2rad(dir_bin_edges[:-1]), freqs, width=dir_width,
               bottom=bottom, color=colours[k],
               label=f'{speed_labels[k]} m/s', align='edge')
        bottom += freqs

    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_ylabel('Frequency (%)', labelpad=30)
    ax.set_title(title, pad=15)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), title='Wind speed')
    return ax


print('Helper functions defined.')

## Task 1: Load and Explore the Data

**What to do:**
1. Set `ROOT_PATH` to the mean-winds data directory and run the cell
2. Inspect basic wind statistics

**Questions:**
- What is the mean wind speed?
- What fraction of the time is the wind speed above 10 m/s (roughly Beaufort force 5)?
- What is the most common wind direction (dominant sector)?

In [ ]:
# ===== EDIT THIS =====
ROOT_PATH = '/gws/ssde/j25a/chil_atmos/wx2026/mean-winds'
QC_GOOD_ONLY = True
PLOT_THEME = 'dark'  # 'light' or 'dark'
# =====================

plt.style.use('dark_background' if PLOT_THEME == 'dark' else 'default')

df = load_wind(ROOT_PATH, qc_good_only=QC_GOOD_ONLY)

print(f'Loaded {len(df):,} records')
print(f'Date range: {df["time"].min().date()} to {df["time"].max().date()}')
print(f'Wind speed (m/s): min={df["speed"].min():.2f}  mean={df["speed"].mean():.2f}  '
      f'max={df["speed"].max():.2f}  std={df["speed"].std():.2f}')
print(f'Fraction > 10 m/s: {(df["speed"] > 10).mean()*100:.1f}%')

## Task 2: Wind Rose

**What to do:**
A **wind rose** shows how often wind comes from each direction, coloured by wind speed.
A `wind_rose()` function has been provided in the setup cell above.

1. Plot a wind rose for the **full dataset**
2. Plot **four wind roses** — one per season (DJF, MAM, JJA, SON) — in a 2×2 grid
3. Describe what you see — does wind direction vary with season?

**Hints:**
- `fig, axes = plt.subplots(2, 2, subplot_kw={'projection': 'polar'}, figsize=(14, 12))`
- Call `wind_rose(spd, drn, ax=ax, title='DJF')` for each season panel
- For UK sites, SW/W winds are the prevailing direction

In [ ]:
# Full-dataset wind rose
wind_rose(df['speed'].values, df['direction'].values, title='Chilbolton — All Data')
plt.tight_layout()
plt.show()

In [ ]:
def season_label(month: int) -> str:
    """Return DJF/MAM/JJA/SON for a month number."""
    # TODO: implement (same as Student 2)
    pass


df['season'] = df['time'].dt.month.map(season_label)

# TODO: create a 2x2 grid of wind roses, one per season
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
# fig, axes = plt.subplots(2, 2, subplot_kw={'projection': 'polar'}, figsize=(14, 12))
# for ax, s in zip(axes.flat, SEASONS):
#     sub = df[df['season'] == s].dropna(subset=['speed', 'direction'])
#     wind_rose(sub['speed'].values, sub['direction'].values, ax=ax, title=s)
# plt.tight_layout()
# plt.show()

## Task 3: Wind Speed Distribution and Exceedance Curve

**What to do:**
1. Plot a histogram of wind speeds (use ~50 bins, and restrict to speeds > 0)
2. Fit a **Weibull distribution** to the wind speed data — this is the standard model for wind speed
3. Compute an exceedance curve: what fraction of the time does wind speed exceed a given value?

**Hints:**
- `scipy.stats.weibull_min.fit(data, floc=0)` fits a Weibull distribution
- Exceedance: sort speeds in descending order; exceedance[i] = (i + 0.5) / n
- Plot on a semi-log y scale (`ax.set_yscale('log')`)

In [ ]:
import scipy.stats

spd_clean = df['speed'].dropna()
spd_clean = spd_clean[spd_clean > 0]  # Exclude calm

# TODO: plot histogram and fit Weibull distribution
# c, loc, scale = scipy.stats.weibull_min.fit(spd_clean, floc=0)
# print(f'Weibull parameters: shape={c:.2f}, scale={scale:.2f} m/s')

# TODO: compute and plot exceedance curve
# spd_sorted = np.sort(spd_clean.values)[::-1]
# n = len(spd_sorted)
# exc = (np.arange(n) + 0.5) / n
# ax.semilogy(spd_sorted, exc)

## Task 4: Monthly Climatology

**What to do:**
Investigate the seasonal cycle of wind speed.

1. Compute the climatological monthly mean wind speed and standard deviation
2. Plot mean ± 1 std as a bar or line chart
3. Identify the windiest and calmest months

**Bonus:** Also plot the monthly mean fraction of observations with wind from the SW quadrant
(225°–315°) — does the prevalence of SW winds change through the year?

In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# TODO: compute and plot monthly mean wind speed ± 1 std

# TODO (bonus): fraction of observations with SW wind per month
# df['is_sw'] = (df['direction'] >= 225) & (df['direction'] <= 315)

## Task 5 (Optional): Storm Identification

**What to do:**
Identify storms from the wind speed record.

1. Resample to hourly means
2. Define a storm as any period where hourly mean wind speed exceeds 15 m/s for at least 3 consecutive hours
3. Print a list of all storms found (start time, peak speed, duration)
4. Can you match any of these to named UK storms?

**Hint:** Use rolling windows or group consecutive-above-threshold points:
```python
df_h['above'] = df_h['speed'] > 15
df_h['storm_id'] = (df_h['above'] != df_h['above'].shift()).cumsum()
```

In [ ]:
# Resample to hourly means
df_hourly = df.set_index('time').resample('h')['speed'].mean().reset_index()

# TODO: identify storm periods (sustained speed > 15 m/s for >= 3 hours)
# TODO: print storm list

## Reflection Questions

1. **Prevailing wind**: What is the dominant wind direction at Chilbolton? Is this consistent with what you would expect for a site in southern England?
2. **Seasonal variation**: Is wind speed higher in winter or summer? By how much?
3. **Speed distribution**: Does the Weibull distribution fit the observed wind speed well? What do the shape and scale parameters tell you?
4. **Comparison with teammates**: Does the wind rose change during periods when Student 3's pressure data shows deep depressions? Do high-wind periods coincide with Student 1's heavy rainfall events?
5. **Extremes**: What is the wind speed exceeded only 1% of the time? How does this compare to the Beaufort scale (https://en.wikipedia.org/wiki/Beaufort_scale)?